# Notebook 00: Environment Setup, Seed Determinism & Artifact Integrity Audit

**FedSocial V2 — Privacy-Preserving Personalized Recommendation with Federated Learning**

This notebook establishes the deterministic execution contract for all 19 research notebooks:
1. Verifies Python, PyTorch, NumPy, Pandas, SciPy, and Scikit-Learn package versions.
2. Enforces global RNG seeds (`SEED = 42`) across Python, NumPy, and PyTorch.
3. Audits on-disk availability and byte sizes of all processed datasets, precomputed embeddings, and benchmark artifacts.

In [1]:
import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Resolve repository root deterministically from ml/notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA_DIR = os.path.join(REPO_ROOT, "ml", "data", "processed")
ARTIFACTS_DIR = os.path.join(REPO_ROOT, "ml", "artifacts")
VIZ_DIR = os.path.join(REPO_ROOT, "ml", "visualizations")
plt.rcParams["figure.dpi"] = 110
import torch, sklearn, scipy

torch.manual_seed(SEED)
env_report = pd.DataFrame([
    {"Component": "Python", "Version": sys.version.split()[0]},
    {"Component": "PyTorch", "Version": torch.__version__},
    {"Component": "NumPy", "Version": np.__version__},
    {"Component": "Pandas", "Version": pd.__version__},
    {"Component": "Scikit-Learn", "Version": sklearn.__version__},
    {"Component": "SciPy", "Version": scipy.__version__},
    {"Component": "Device", "Version": "cuda" if torch.cuda.is_available() else "cpu"},
])
print(env_report.to_string(index=False))

   Component    Version
      Python     3.11.9
     PyTorch 2.15.0+cpu
       NumPy      2.4.6
      Pandas      3.1.1
Scikit-Learn      1.9.1
       SciPy     1.17.1
      Device        cpu


In [2]:
required_files = [
    os.path.join(DATA_DIR, "articles.json"),
    os.path.join(DATA_DIR, "articles_id.json"),
    os.path.join(DATA_DIR, "interactions.csv"),
    os.path.join(DATA_DIR, "article_embeddings.npy"),
    os.path.join(DATA_DIR, "federated_partitions.json"),
    os.path.join(ARTIFACTS_DIR, "federated_benchmark.json"),
]
rows = []
for p in required_files:
    exists = os.path.exists(p)
    size_kb = os.path.getsize(p) / 1024 if exists else 0.0
    rows.append({"Artifact": os.path.relpath(p, REPO_ROOT), "Exists": exists, "Size_KB": round(size_kb, 2)})

audit_df = pd.DataFrame(rows)
assert audit_df["Exists"].all(), "Missing required ML data artifact!"
print(audit_df.to_string(index=False))
print("\nAll core datasets and benchmark artifacts verified successfully.")

                                   Artifact  Exists  Size_KB
            ml\data\processed\articles.json    True 18611.28
         ml\data\processed\articles_id.json    True  2920.32
         ml\data\processed\interactions.csv    True   814.63
   ml\data\processed\article_embeddings.npy    True  4500.12
ml\data\processed\federated_partitions.json    True     8.75
      ml\artifacts\federated_benchmark.json    True     6.38

All core datasets and benchmark artifacts verified successfully.
